### Import libraries

This cell imports pandas for reading and working with tabular data, and NumPy for numerical operations and missing-value handling.

In [12]:
import pandas as pd
import numpy as np

## 1. Imports and configuration

This cell documents the notebook configuration. The paths identify the four raw datasets, and the sentinel values identify codes that should become missing values.

In [13]:
from pathlib import Path

DATA_DIR = Path('../data')
MISSING_VALUES = [-666666666, -6666666666]
ZIP_PATTERN = r'\d{5}'

DATASETS = {
    'public_features': DATA_DIR / 'miami_dade_public_features.csv',
    'area_features': DATA_DIR / 'area_features.csv',
    'area_options': DATA_DIR / 'area_options.csv',
    'crowd_text_snippets': DATA_DIR / 'crowd_text_snippets.csv',
}

## 2. Reusable functions

These functions keep the cleaning rules consistent across datasets. ZIP codes are handled as strings, missing sentinel values are converted to `NaN`, ordinary categorical columns are one-hot encoded, and comma-separated tags are encoded separately.

In [14]:
def load_dataset(path):
    """Load a CSV while preserving ZIP codes as strings."""
    return pd.read_csv(path, dtype={'zip': 'string'})


def replace_missing_values(df, missing_values=MISSING_VALUES):
    """Replace configured sentinel values with pandas missing values."""
    return df.replace(missing_values, np.nan)


def validate_zip_codes(df, zip_column='zip'):
    """Keep rows whose ZIP value is present and exactly five digits."""
    cleaned_df = df.copy()
    zip_values = cleaned_df[zip_column].astype('string').str.strip()
    valid_zip = zip_values.notna() & zip_values.str.fullmatch(ZIP_PATTERN).fillna(False)
    cleaned_df[zip_column] = zip_values
    return cleaned_df.loc[valid_zip].copy()


def one_hot_encode(df, columns):
    """One-hot encode selected categorical columns and remove their originals."""
    existing_columns = [column for column in columns if column in df.columns]
    return pd.get_dummies(df, columns=existing_columns, dtype=int)


def one_hot_encode_tags(df, column='tags'):
    """Split comma-separated tags and create one binary column per tag."""
    if column not in df.columns:
        return df.copy()

    tags = df[column].fillna('').astype('string').str.get_dummies(sep=',')
    tags.columns = [f'tag_{tag.strip()}' for tag in tags.columns]
    tags = tags.loc[:, tags.columns != 'tag_']
    return pd.concat([df.drop(columns=[column]), tags], axis=1)

## 3. Load all four datasets

This cell loads the public features, area features, area options, and crowd text datasets. The reusable loader preserves the `zip` column as text so leading zeroes are not lost.

In [15]:
datasets = {
    name: load_dataset(path)
    for name, path in DATASETS.items()
}

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

## 4. Display original shapes and schemas

This cell records each dataset's size, column names, and data types before cleaning. This gives us a baseline for comparing the cleaned outputs.

In [16]:
original_shapes = {}

for name, dataset in datasets.items():
    original_shapes[name] = dataset.shape
    print(f'\n{name}: shape={dataset.shape}')
    print('columns:', dataset.columns.tolist())
    print(dataset.dtypes)


public_features: shape=(80, 17)
columns: ['NAME', 'B01003_001E', 'B25064_001E', 'B25077_001E', 'B07003_001E', 'B07003_004E', 'B07003_008E', 'B07003_010E', 'B11016_001E', 'B25003_002E', 'B25003_003E', 'zip', 'moved_different_house_pct', 'median_rent_usd', 'population', 'zillow_home_value_index', 'zillow_rent_index']
NAME                                 object
B01003_001E                           int64
B25064_001E                           int64
B25077_001E                           int64
B07003_001E                           int64
B07003_004E                           int64
B07003_008E                           int64
B07003_010E                           int64
B11016_001E                           int64
B25003_002E                           int64
B25003_003E                           int64
zip                          string[python]
moved_different_house_pct           float64
median_rent_usd                       int64
population                            int64
zillow_home_value_inde

## 5. Replace missing sentinel values

This cell applies the same reusable missing-value function to all four datasets. Both sentinel spellings are handled, and the replacement counts are printed for auditing.

In [17]:
sentinel_counts = {}

for name, dataset in datasets.items():
    sentinel_mask = dataset.isin(MISSING_VALUES)
    sentinel_counts[name] = int(sentinel_mask.to_numpy().sum())
    datasets[name] = replace_missing_values(dataset)
    print(f'{name}: replaced {sentinel_counts[name]} sentinel values')

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

public_features: replaced 8 sentinel values
area_features: replaced 3 sentinel values
area_options: replaced 0 sentinel values
crowd_text_snippets: replaced 0 sentinel values


## 6. Validate ZIP columns

This cell applies the reusable ZIP validation function to every dataset. Rows with an empty, missing, or non-five-digit ZIP are removed, and the number of retained rows is reported.

In [18]:
zip_validation_counts = {}

for name, dataset in datasets.items():
    validated_dataset = validate_zip_codes(dataset)
    zip_validation_counts[name] = {
        'before': len(dataset),
        'after': len(validated_dataset),
        'dropped': len(dataset) - len(validated_dataset),
    }
    datasets[name] = validated_dataset
    print(f'{name}: kept {len(validated_dataset)} of {len(dataset)} rows')

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

public_features: kept 80 of 80 rows
area_features: kept 80 of 80 rows
area_options: kept 10 of 10 rows
crowd_text_snippets: kept 20 of 20 rows


## 7. Apply one-hot encoding

This cell encodes categorical columns while leaving identifiers, ZIP codes, and free text unchanged. Tags are split into separate binary columns because one row can contain multiple tags.

In [23]:
area_features = one_hot_encode(area_features, ['rent_band'])
area_options = one_hot_encode(area_options, ['rent_band', 'housing_preference', 'household'])
area_options = one_hot_encode_tags(area_options, 'tags')
crowd_text_snippets = one_hot_encode(crowd_text_snippets, ['theme'])

datasets.update({
    'public_features': public_features,
    'area_features': area_features,
    'area_options': area_options,
    'crowd_text_snippets': crowd_text_snippets,
})

for name, dataset in datasets.items():
    print(f'\n{name} after one-hot encoding:')
    print(dataset.head())
    print('columns:', dataset.columns.tolist())


public_features after one-hot encoding:
          NAME  B01003_001E  B25064_001E  B25077_001E  B07003_001E  \
0  ZCTA5 33101            0          NaN          NaN            0   
1  ZCTA5 33109          792          NaN    2000001.0          792   
2  ZCTA5 33122         1873       2370.0    1335900.0         1824   
3  ZCTA5 33125        54873       1238.0     339300.0        54290   
4  ZCTA5 33126        46798       1546.0     287100.0        46403   

   B07003_004E  B07003_008E  B07003_010E  B11016_001E  B25003_002E  \
0            0            0            0            0            0   
1          630           97            0          328          307   
2          962          241            8          816           76   
3        49530         2057          171        21227         5080   
4        41635         1763          240        18073         7104   

   B25003_003E    zip  moved_different_house_pct  median_rent_usd  population  \
0            0  33101               

## 8. Print cleaned results

This cell prints a preview and the shape of every cleaned dataset so we can inspect the shared transformations and confirm that one-hot columns were added.

In [20]:
for name, dataset in datasets.items():
    print(f'\n{name}: shape={dataset.shape}')
    display(dataset.head())


public_features: shape=(80, 17)


,NAME,B01003_001E,B25064_001E,B25077_001E,B07003_001E,B07003_004E,B07003_008E,B07003_010E,B11016_001E,B25003_002E,B25003_003E,zip,moved_different_house_pct,median_rent_usd,population,zillow_home_value_index,zillow_rent_index
0,ZCTA5 33101,0,NaN,NaN,0,0,0,0,0,0,0,33101,NaN,NaN,0,NaN,NaN
1,ZCTA5 33109,792,NaN,2000001.0,792,630,97,0,328,307,21,33109,0.204545,NaN,792,6.413596e+06,NaN
2,ZCTA5 33122,1873,2370.0,1335900.0,1824,962,241,8,816,76,740,33122,0.472588,2370.0,1873,1.505538e+06,2740.611111
3,ZCTA5 33125,54873,1238.0,339300.0,54290,49530,2057,171,21227,5080,16147,33125,0.087677,1238.0,54873,4.597721e+05,2313.579167
4,ZCTA5 33126,46798,1546.0,287100.0,46403,41635,1763,240,18073,7104,10969,33126,0.102752,1546.0,46798,3.017829e+05,2380.241026



area_features: shape=(80, 16)


,zip,area_name,transit,social,quiet,pet_friendly,walkable,migration_score,median_rent_usd,co_living_friendly,single_family_friendly,rent_band_1200-1800,rent_band_1800-2200,rent_band_2200-2800,rent_band_2800-3500,rent_band_3500+
0,33101,ZCTA5 33101,0.000000,0.000000,1.000000,0.6,0.000000,0.000000,NaN,0.150000,0.650000,1,0,0,0,0
1,33109,ZCTA5 33109,0.000000,0.011141,0.994429,0.6,0.000000,0.432820,NaN,0.179813,0.650000,1,0,0,0,0
2,33122,ZCTA5 33122,0.461831,0.026348,0.986826,0.6,0.999998,1.000000,2370.0,0.415809,0.650000,0,0,1,0,0
3,33125,Riverside,0.883777,0.771902,0.614049,0.6,0.999997,0.185526,1238.0,0.537352,0.495454,1,0,0,0,0
4,33126,ZCTA5 33126,0.707822,0.658311,0.670845,0.6,0.999997,0.217424,1546.0,0.481956,0.557929,1,0,0,0,0



area_options: shape=(10, 17)


,option_id,zip,summary,disclaimer,rent_band_1200-1800,rent_band_1800-2200,rent_band_2200-2800,housing_preference_co_living,housing_preference_own_apartment,household_alone,household_with_co_leaser,tag_co_living,tag_pet_friendly,tag_quiet,tag_social,tag_transit,tag_walkable
0,SYN-33127-OWN,33127,Wynwood (ZIP 33127) — your own apartment (e.g....,NOT A REAL LISTING — synthetic demo only.,1,0,0,0,1,0,1,0,0,0,1,1,1
1,SYN-33128-OWN,33128,Downtown Miami (ZIP 33128) — your own apartmen...,NOT A REAL LISTING — synthetic demo only.,1,0,0,0,1,0,1,0,0,0,0,1,1
2,SYN-33130-OWN,33130,Brickell (ZIP 33130) — your own apartment with...,NOT A REAL LISTING — synthetic demo only.,1,0,0,0,1,0,1,0,0,0,1,1,1
3,SYN-33127-OWN-SOLO,33127,"Wynwood (ZIP 33127) — your own studio or 1BR, ...",NOT A REAL LISTING — synthetic demo only.,1,0,0,0,1,1,0,0,0,0,1,1,1
4,SYN-33128-COLIV,33128,Downtown Miami (ZIP 33128) — co-living option ...,NOT A REAL LISTING — synthetic co-living demo ...,1,0,0,1,0,1,0,1,0,0,1,1,0



crowd_text_snippets: shape=(20, 11)


,snippet_id,zip,source,text,theme_co_living,theme_expensive,theme_pet_friendly,theme_quiet,theme_single_family,theme_social,theme_transit
0,T1,33127,kaggle_reviews,Great walkability and easy bus access; felt we...,0,0,0,0,0,0,1
1,T2,33127,kaggle_reviews,Lots of events and people meeting up; can be l...,0,0,0,0,0,1,0
2,T3,33130,kaggle_reviews,Metrorail nearby made commuting easy without a...,0,0,0,0,0,0,1
3,T4,33130,kaggle_reviews,Busy area; street noise is real in some towers.,0,0,0,1,0,0,0
4,T5,33139,kaggle_reviews,Very social neighborhood; beach culture and ni...,0,0,0,0,0,1,0


In [21]:
for name, dataset in datasets.items():
    remaining_sentinels = int(dataset.isin(MISSING_VALUES).to_numpy().sum())
    valid_zip = dataset['zip'].notna() & dataset['zip'].str.fullmatch(ZIP_PATTERN).fillna(False)
    duplicate_columns = dataset.columns[dataset.columns.duplicated()].tolist()

    assert remaining_sentinels == 0, f'{name} still contains sentinel values'
    assert valid_zip.all(), f'{name} contains an invalid ZIP code'
    assert not duplicate_columns, f'{name} contains duplicate columns: {duplicate_columns}'

    print(f'{name}: validation passed; rows={len(dataset)}, columns={len(dataset.columns)}')

public_features: validation passed; rows=80, columns=17
area_features: validation passed; rows=80, columns=16
area_options: validation passed; rows=10, columns=17
crowd_text_snippets: validation passed; rows=20, columns=11


## 9. Run final validation checks

This cell verifies that no configured sentinel values remain, all ZIP values are valid five-digit strings, and the cleaned datasets have no duplicate column names.

## 10. Save cleaned datasets

This cell writes each cleaned DataFrame to a new CSV with `_cleaned` in the filename. The original raw datasets are not overwritten.